In [30]:
# Импорт класса SparkSession для работы с DataFrame и SQL API
from pyspark.sql import SparkSession

# Создание или подключение к существующей SparkSession
spark = (
    SparkSession.builder
        .appName("L1_Apache_Spark")              # Имя приложения в интерфейсе Spark
        .master("local[4]")                      # Локальный режим исполнения на 4 ядрах
        .config("spark.executor.memory", "2g")   # Объём памяти на каждого исполнителя
        .config("spark.driver.memory", "2g")     # Объём памяти, выделяемый драйверу
        .config("spark.python.worker.timeout", "12000")  # Таймаут для Python-воркеров (в секундах)
        .getOrCreate()                           # Возвращает существующую сессию или создаёт новую
)

# Извлечение SparkContext для работы с RDD и установки уровня логирования
sc = spark.sparkContext
sc.setLogLevel("WARN")  # Уровень WARN фильтрует информационные и отладочные сообщения


In [37]:
# -----------------------------------------------------------------------------
# Загружаем CSV в RDD, исключаем заголовок и сохраняем в кэше
# -----------------------------------------------------------------------------
trips = (
    sc.textFile("trips.csv")
      .map(lambda line: line.split(",", -1))
      .filter(lambda cols: cols[0] != "trip_id")  # убираем шапку по первому полю
)

stations = (
    sc.textFile("stations.csv")
      .map(lambda line: line.split(",", -1))
      .filter(lambda cols: cols[0] != "station_id")
)


1.	Найти велосипед с максимальным временем пробега.

In [39]:
# -----------------------------------------------------------------------------
# Парсинг строк в объекты Trip
# -----------------------------------------------------------------------------
from typing import NamedTuple
from datetime import datetime

class Trip(NamedTuple):
    """NamedTuple для хранения всех полей одной поездки"""
    trip_id: int
    duration: int
    start_date: datetime
    start_station_name: str
    start_station_id: int
    end_date: datetime
    end_station_name: str
    end_station_id: int
    bike_id: int
    subscription_type: str
    zip_code: str


def parse_trips_partition(records):
    """
    Преобразуем партицию строк (list of lists) в Trip-объекты.
    Используем try/except, чтобы пропускать записи с повреждёнными данными.
    Плюс мы инициализируем NamedTuple один раз на партицию, а не per-row.
    """
    for rec in records:
        try:
            yield Trip(
                trip_id=int(rec[0]),
                duration=int(rec[1]),
                start_date=datetime.strptime(rec[2], '%m/%d/%Y %H:%M'),
                start_station_name=rec[3],
                start_station_id=int(rec[4]),
                end_date=datetime.strptime(rec[5], '%m/%d/%Y %H:%M'),
                end_station_name=rec[6],
                end_station_id=int(rec[7]),
                bike_id=int(rec[8]),
                subscription_type=rec[9],
                zip_code=rec[10]
            )
        except (ValueError, IndexError):
            # Пропускаем некорректные или неполные записи
            continue

trips_parsed = trips.mapPartitions(parse_trips_partition).cache()

# -----------------------------------------------------------------------------
# Вычисление суммарного времени пробега по каждому байку
# -----------------------------------------------------------------------------
#  Преобразуем в пары (bike_id, duration)
#  Суммируем с помощью reduceByKey — оптимизированный метод для commutative ops
bike_duration_pairs = trips_parsed.map(lambda trip: (trip.bike_id, trip.duration))
duration_by_bike = bike_duration_pairs.reduceByKey(lambda acc, dur: acc + dur)

# -----------------------------------------------------------------------------
# Поиск велосипеда с максимальным общим временем пробега
# -----------------------------------------------------------------------------
# reduce выберет кортеж с наибольшим значением total_duration
max_bike_id, max_total_duration = duration_by_bike.reduce(
    lambda x, y: x if x[1] > y[1] else y
)

# -----------------------------------------------------------------------------
# Вывод результата
# -----------------------------------------------------------------------------
print(
    f"Bike with max total duration: ID={max_bike_id}, Total duration={max_total_duration} sec"
)

Bike with max total duration: ID=535, Total duration=18611693 sec


2.	Найти наибольшее геодезическое расстояние между станциями.

In [40]:
from geopy.distance import geodesic
import itertools

# 1) Инициализация Spark
spark = (
    SparkSession.builder
        .appName("MaxStationDistance")  # Отображается в Spark UI
        .master("local[4]")              # Работа на 4 локальных ядрах
        .config("spark.executor.memory", "2g")
        .config("spark.driver.memory", "2g")
        .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("WARN")  # Только предупреждения и ошибки

# 2) Определяем NamedTuple для станции и парсим RDD строк в объекты Station
class Station(NamedTuple):
    station_id: int
    name: str
    lat: float
    long: float
    dockcount: int
    landmark: str
    installation: datetime


def parse_stations_partition(records):
    """
    Преобразуем строки CSV в объекты Station.
    Инициализация NamedTuple только раз на партицию через mapPartitions.
    """
    for rec in records:
        try:
            yield Station(
                station_id=int(rec[0]),
                name=rec[1],
                lat=float(rec[2]),
                long=float(rec[3]),
                dockcount=int(rec[4]),
                landmark=rec[5],
                installation=datetime.strptime(rec[6], '%m/%d/%Y')
            )
        except (ValueError, IndexError):
            # Пропускаем некорректные или неполные строки
            continue

# Загружаем CSV, убираем заголовок и парсим в Station; кешируем для эффекта
stations_rdd = (
    sc.textFile("stations.csv")
      .map(lambda line: line.split(',', -1))
      .filter(lambda cols: cols[0] != 'station_id')
      .mapPartitions(parse_stations_partition)
      .cache()
)

# 3) Сборка списка станций в драйвере для локальной комбинации
# Предполагаем, что количество станций невелико и помещается в память
stations_list = stations_rdd.map(lambda s: (s.station_id, (s.lat, s.long))).collect()

# 4) Локальный перебор пар комбинаций через itertools.combinations
# Вычисляем геодезическое расстояние и находим максимум
max_dist = 0.0
max_pair = (None, None)
for (id1, coord1), (id2, coord2) in itertools.combinations(stations_list, 2):
    dist = geodesic(coord1, coord2).km
    if dist > max_dist:
        max_dist = dist
        max_pair = (id1, id2)

# 5) Вывод результата
print(f"Max geodesic distance: {max_dist:.2f} km between stations {max_pair[0]} and {max_pair[1]}")


Max geodesic distance: 69.92 km between stations 16 and 60


3.	Найти путь велосипеда с максимальным временем пробега через станции.

In [41]:
# 1) Фильтруем только нужный байк, затем извлекаем (дата старта, станция старта, станция финиша)
path_rdd = (
    tripsByBike
      .filter(lambda kv: kv[0] == maxDurationBike[0])  # оставляем только записи по целевому bike_id
      .map(lambda kv: (kv[1].start_date, kv[1].start_station_name, kv[1].end_station_name))
      .sortBy(lambda rec: rec[0])                       # сортируем по времени старта поездки
)

# 2) Собираем упорядоченный список этапов маршрута на драйвере
#    для каждого шага берем имя станции старта, а в конце добавляем последнюю станцию финиша
checkpoints = path_rdd.collect()
stations_path = [start for (_, start, _) in checkpoints]
stations_path.append(checkpoints[-1][2])

# 3) Выводим итоговый маршрут
print(f"Path of bike {maxDurationBike[0]}: {stations_path}")

Path of bike 535: ['Post at Kearney', 'San Francisco Caltrain (Townsend at 4th)', 'San Francisco Caltrain 2 (330 Townsend)', 'Market at Sansome', '2nd at Townsend', 'San Francisco City Hall', 'Civic Center BART (7th at Market)', 'Post at Kearney', 'Embarcadero at Sansome', 'Washington at Kearney', 'Market at Sansome', 'Market at Sansome', '2nd at Folsom', 'Temporary Transbay Terminal (Howard at Beale)', '2nd at Townsend', 'Embarcadero at Sansome', 'Clay at Battery', 'Harry Bridges Plaza (Ferry Building)', 'Clay at Battery', 'San Francisco Caltrain (Townsend at 4th)', 'Steuart at Market', '2nd at Townsend', 'Harry Bridges Plaza (Ferry Building)', 'Townsend at 7th', 'San Francisco Caltrain 2 (330 Townsend)', 'San Francisco Caltrain 2 (330 Townsend)', 'Steuart at Market', 'San Francisco Caltrain (Townsend at 4th)', '2nd at South Park', 'Post at Kearney', '2nd at Folsom', 'Mechanics Plaza (Market at Battery)', 'Powell at Post (Union Square)', 'Powell at Post (Union Square)', 'Powell at Pos

4.	Найти количество велосипедов в системе.

In [43]:
# Используем distinct() для поиска уникальных bike_id, после чего считаем их количество
bikesCount = tripsByBike.keys().distinct().count()

# Выводим итоговое количество велосипедов
print(f"Total number of bikes: {bikesCount}")

Total number of bikes: 700


5.	Найти пользователей потративших на поездки более 3 часов.

In [44]:
# Группируем поездки по пользователям (используя zip_code как идентификатор пользователя)
tripsByUsers = tripsInternal.keyBy(lambda trip: trip.zip_code)

# Суммируем общее время поездок каждого пользователя
# Фильтруем тех, чья сумма поездок превышает 3 часа (10800 секунд)
usersWith3HourLongTrips = (
    tripsByUsers
    .aggregateByKey(
        0,                                          # Начальное значение аккумулятора
        lambda acc, trip: acc + trip.duration,      # Функция суммирования внутри партиции
        lambda lhs, rhs: lhs + rhs                  # Функция суммирования результатов между партициями
    )
    .filter(lambda t: t[1] > 10800)                # Оставляем только пользователей с суммой > 3 часов
    .keys()                                        # Извлекаем только идентификаторы пользователей
    .collect()                                     # Собираем результат на драйвере
)

# Выводим итоговый список пользователей
print(f"Users with more than 3 hours of total trips: {usersWith3HourLongTrips}")

Users with more than 3 hours of total trips: ['95060', '95112', '94041', '94117', '94402', '94102', '94612', '94609', '94158', '94133', '94597', '', '94121', '95118', '94610', '95136', '2142', '94703', '95070', '94404', '94518', '94549', '94556', '94805', '95014', '97330', '94005', '92178', '85008', '94606', '94941', '94901', '94577', '94523', '92111', '95618', '89052', '94014', '10025', '78230', '10022', '95111', '75201', '94141', '90046', '34110', '1945', '75225', '90032', '4517', '94080', '95148', '92808', '63130', '89448', '94539', '90024', '20008', '19803', '91605', '10036', '90049', '91214', '5024', '90291', '34990', '91801', '94928', '92037', '16801', '95003', '95472', '92109', '90025', '94952', '11530', '91748', '95351', '98122', '10044', '84604', '93041', '94568', '1742', '95121', '92805', '21202', '91206', '95120', '94304', '93109', '94130', '11570', '91343', '95814', '91711', '90278', '10065', '95128', '94042', '93405', '94590', '94506', '60514', '90026', '92618', '95062', '